# Bat_Transformer vs. attention-ablated baseline vs. GLM family

Compares five encoders on the same train/valid/test splits used in
`preprocess-data-v0j-share.ipynb`:

1. **Transformer** — full `Bat_Transformer` with RoPE attention.
2. **Transformer (no attention)** — same architecture with `use_attention=False`,
   so each `TransformerEncoderRoPE` block is replaced by a per-timestep
   `_PerTimestepFFN` (LayerNorm + GELU FFN + residual). No temporal mixing,
   no positional info — RoPE is dropped along with the attention.
3. **GLM** — `GLM_3DCosineBasis`, position-only (lag=0).
4. **GLM + lags** — same with time-lagged positions in `[-1, +1] s`.
5. **GLM + lags + kinematics** — adds speed + velocity-direction bumps + IMU
   (uses the kinematics dataloader: `use_speed_head_dir=True, use_IMU_flag=True`).

Comparison metrics: per-neuron Poisson FDE and Pearson correlation on the
held-out test set, broken down by flight filter (all / in-domain / OOD).

In [14]:
%load_ext autoreload
%autoreload 2

import os
import sys
from copy import deepcopy

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from torch import nn
from scipy.stats import wilcoxon

sys.path.append(os.path.expanduser('~/Documents/code/digitalbat'))
from digitalbat.datasets.data import bat_data_loader
from digitalbat.datasets.defaults import DEFAULT_SESSION_NAMES, make_data_config
from digitalbat.models.former import Bat_Transformer
from digitalbat.models.former_defaults import (
    DEFAULT_LR, DEFAULT_WEIGHT_DECAY,
    make_model_config, make_model_name, make_trainer_config,
)
from digitalbat.models.glm import (
    GLM_3DCosineBasis, GLM_3DCosineBasis_Kinematics,
)
from digitalbat.models.layers import ELU1
from digitalbat.training.trainer import bat_former_trainer
from digitalbat.utility.utils import (
    corr, poisson_fde_population, set_random_state,
    bat_palette, parse_bat, wilcoxon_safe, add_significance_bracket,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
seed = 1
set_random_state(seed)
device = 'cuda'

# Canonical model order (used for plotting / DataFrame column order).
MODEL_ORDER = ['GLM', 'GLM + lags', 'GLM + lags + kin',
               'Transformer (no attn)', 'Transformer']

# Where the pretrained checkpoints live. Defaults match
# preprocess-data-v0j-share.ipynb's canonical paths; override here if different.
BAT_FORMER_CKPT = make_model_name(
    seed, DEFAULT_LR, DEFAULT_WEIGHT_DECAY,
    out_dir='./out/bat_former', suffix='',
)
BAT_FORMER_NOATTN_CKPT = make_model_name(
    seed, DEFAULT_LR, DEFAULT_WEIGHT_DECAY,
    out_dir='./out/bat_former_no_attn', suffix='',
)
os.makedirs(os.path.dirname(BAT_FORMER_NOATTN_CKPT), exist_ok=True)

# GLM checkpoint paths use the tag pattern from the reference notebook:
#   ./out/bat_glm/bat_{tag}_seed_{seed}_wd_{wd}.pth
GLM_WEIGHT_DECAYS = {'GLM': 1.0, 'GLM + lags': 10.0, 'GLM + lags + kin': 10.0}

def _glm_ckpt(name, wd):
    tag = name.lower().replace(' + ', '_').replace(' ', '_')
    return f'./out/bat_glm/bat_{tag}_seed_{seed}_wd_{wd}.pth'

CKPT_PATHS = {
    'Transformer': BAT_FORMER_CKPT,
    'Transformer (no attn)': BAT_FORMER_NOATTN_CKPT,
    'GLM': _glm_ckpt('GLM', GLM_WEIGHT_DECAYS['GLM']),
    'GLM + lags': _glm_ckpt('GLM + lags', GLM_WEIGHT_DECAYS['GLM + lags']),
    'GLM + lags + kin': _glm_ckpt('GLM + lags + kin', GLM_WEIGHT_DECAYS['GLM + lags + kin']),
}
for name, path in CKPT_PATHS.items():
    print(f'  {name:<22} -> {path}  (exists={os.path.exists(path)})')

BAT_PALETTE = bat_palette()

  Transformer            -> ./out/bat_former/bat_former_seed_1_lr_1.00e-02_wd_1.00e-02.pth  (exists=True)
  Transformer (no attn)  -> ./out/bat_former_no_attn/bat_former_seed_1_lr_1.00e-02_wd_1.00e-02.pth  (exists=True)
  GLM                    -> ./out/bat_glm/bat_glm_seed_1_wd_1.0.pth  (exists=True)
  GLM + lags             -> ./out/bat_glm/bat_glm_lags_seed_1_wd_10.0.pth  (exists=True)
  GLM + lags + kin       -> ./out/bat_glm/bat_glm_lags_kin_seed_1_wd_10.0.pth  (exists=True)


## Data loader

Single call to `bat_data_loader` shared by both models so the train/valid/test
splits and the standardization stats are identical.

In [16]:
data_config = make_data_config()
dataloaders, behavior_dim, unit_num_dict, meta_info_dict = bat_data_loader(**data_config)
print(f'Base dataloader: behavior_dim={behavior_dim}, {len(unit_num_dict)} sessions')

# Kinematics dataloader for the `GLM + lags + kin` variant: adds speed,
# head direction (vel_unit), and IMU to the behavior vector
# (dims: 0-2=xyz, 3=speed, 4-6=vel_unit, 7-9=imu_acc, 10-12=imu_gyr).
data_config_kin = {**data_config, 'use_speed_head_dir': True, 'use_IMU_flag': True}
dataloaders_kin, behavior_dim_kin, unit_num_dict_kin, meta_info_dict_kin = bat_data_loader(**data_config_kin)
print(f'Kinematics dataloader: behavior_dim={behavior_dim_kin}')

# Per-model (dataloaders, meta_info_dict) routing.
DATA_BY_MODEL = {
    'Transformer':            (dataloaders,     meta_info_dict),
    'Transformer (no attn)':  (dataloaders,     meta_info_dict),
    'GLM':                    (dataloaders,     meta_info_dict),
    'GLM + lags':             (dataloaders,     meta_info_dict),
    'GLM + lags + kin':       (dataloaders_kin, meta_info_dict_kin),
}

  Acceleration filter: removed 4/37 flights (peak |acc| > 40 m/s²)
  Acceleration filter: removed 4/136 flights (peak |acc| > 40 m/s²)
  Acceleration filter: removed 14/154 flights (peak |acc| > 40 m/s²)
  Acceleration filter: removed 6/131 flights (peak |acc| > 40 m/s²)
  Acceleration filter: removed 2/79 flights (peak |acc| > 40 m/s²)
  Acceleration filter: removed 2/78 flights (peak |acc| > 40 m/s²)
  Acceleration filter: removed 1/77 flights (peak |acc| > 40 m/s²)
  Acceleration filter: removed 4/82 flights (peak |acc| > 40 m/s²)
  Acceleration filter: removed 1/98 flights (peak |acc| > 40 m/s²)
  Acceleration filter: removed 3/80 flights (peak |acc| > 40 m/s²)
  Acceleration filter: removed 4/112 flights (peak |acc| > 40 m/s²)
  Acceleration filter: removed 2/145 flights (peak |acc| > 40 m/s²)
Base dataloader: behavior_dim=3, 18 sessions
  Acceleration filter: removed 4/37 flights (peak |acc| > 40 m/s²)
  Acceleration filter: removed 4/136 flights (peak |acc| > 40 m/s²)
  Accelera

## Models

Build all five encoders. The two `Bat_Transformer` variants share `make_model_config`
(only `use_attention` differs). The three GLM variants follow the same configs as
`preprocess-data-v0j-share.ipynb`.

In [17]:
# All five models use the full 18-session unit_num_dict (kin uses the kinematics
# variant). If an on-disk GLM checkpoint was trained on a subset, the train cell
# below will detect the key mismatch and retrain (overwriting the old ckpt).
glm_common = dict(
    n_bases_per_dim=16, pos_range=(-3.0, 3.0),
    width_to_spacing_ratio=4, activation='exp',
    bin_width=data_config['bin_width'],
)
glm_lags = np.arange(-1.0, 1.0 + 1e-9, 0.2).tolist()

glm_configs_by_name = {
    'GLM': (GLM_3DCosineBasis,
            dict(**glm_common, output_dim_dict=unit_num_dict)),
    'GLM + lags': (GLM_3DCosineBasis,
                   dict(**glm_common, output_dim_dict=unit_num_dict,
                        lag_seconds=glm_lags)),
    'GLM + lags + kin': (GLM_3DCosineBasis_Kinematics, dict(
        output_dim_dict=unit_num_dict_kin,
        behavior_stats={s: (meta_info_dict_kin['behavior_mean'][s],
                            meta_info_dict_kin['behavior_std'][s])
                        for s in meta_info_dict_kin['behavior_mean']},
        n_bases_per_dim=16, pos_range=(-3.0, 3.0), width_to_spacing_ratio=4,
        activation='exp', bin_width=data_config['bin_width'],
        lag_seconds=glm_lags, n_vel_bumps=8, vel_width_ratio=4,
    )),
}

base_model_config = make_model_config(behavior_dim, unit_num_dict, meta_info_dict)
transformer_noattn_cfg = {**base_model_config, 'use_attention': False}

models = {
    'Transformer':           Bat_Transformer(**base_model_config),
    'Transformer (no attn)': Bat_Transformer(**transformer_noattn_cfg),
}
for name in ['GLM', 'GLM + lags', 'GLM + lags + kin']:
    cls, cfg = glm_configs_by_name[name]
    models[name] = cls(**cfg)

def _count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)
for name in MODEL_ORDER:
    print(f'  {name:<22}: {_count_params(models[name]):>10,} parameters')

def _ckpt_is_compatible(model, path):
    """Return True iff loading `path` into `model` would succeed under strict=True."""
    if not os.path.exists(path):
        return False
    try:
        state = torch.load(path, map_location='cpu', weights_only=False)
    except Exception:
        return False
    model_keys = set(model.state_dict().keys())
    ckpt_keys = set(state.keys())
    if model_keys != ckpt_keys:
        return False
    for k in model_keys:
        if model.state_dict()[k].shape != state[k].shape:
            return False
    return True

ckpt_compat = {name: _ckpt_is_compatible(models[name], CKPT_PATHS[name]) for name in MODEL_ORDER}
print()
for name in MODEL_ORDER:
    print(f'  {name:<22}: ckpt compatible = {ckpt_compat[name]}')

GLM: 11 lags x 4096 spatial = 45056 features, lag_bins=[-10, -8, -6, -4, -2, 0, 2, 4, 6, 8, 10]
GLM Kinematics: 11 lags x (4096 spatial + 24 vel + 7 speed+IMU) = 45397 features
  GLM                   : 13,802,793 parameters
  GLM + lags            : 151,797,033 parameters
  GLM + lags + kin      : 152,945,862 parameters
  Transformer (no attn) :    113,481 parameters
  Transformer           :    117,769 parameters

  GLM                   : ckpt compatible = False
  GLM + lags            : ckpt compatible = False
  GLM + lags + kin      : ckpt compatible = False
  Transformer (no attn) : ckpt compatible = True
  Transformer           : ckpt compatible = True


## Train missing checkpoints

The pretrained `Transformer` and `GLM*` checkpoints already exist on disk under
`./out/bat_former/` and `./out/bat_glm/` (the GLM ones were trained via
`notebooks/null_distribution_training.py`). The ablated `Transformer (no attn)`
checkpoint is the only one this notebook trains from scratch — but the cell will
also (re)train any GLM whose checkpoint is missing, using the same
`bat_former_trainer` and best `weight_decay` values from the reference notebook.

In [18]:
# --- Train Transformer (no attn) if its checkpoint is missing or incompatible ---
if ckpt_compat['Transformer (no attn)']:
    print(f'[Transformer (no attn)] checkpoint exists and is compatible — skipping training.')
else:
    print(f'[Transformer (no attn)] training fresh ...')
    set_random_state(seed)
    models['Transformer (no attn)'] = Bat_Transformer(**transformer_noattn_cfg).to(device)
    trainer_config = make_trainer_config(
        model=models['Transformer (no attn)'],
        dataloaders=dataloaders, device=device,
        model_name=BAT_FORMER_NOATTN_CKPT,
        lr=DEFAULT_LR, weight_decay=DEFAULT_WEIGHT_DECAY,
    )
    best_acc, _, acc_curve = bat_former_trainer(**trainer_config)
    print(f'  best val FDE (median): {best_acc:.4f}')
    plt.figure(figsize=(4, 3))
    plt.plot(acc_curve)
    plt.xlabel('Epoch'); plt.ylabel('Val FDE (median)')
    plt.title('Transformer (no attn) training curve'); plt.tight_layout(); plt.show()

# --- Train any missing or incompatible GLM checkpoints (overwrites old subset-trained ckpts) ---
for name in ['GLM', 'GLM + lags', 'GLM + lags + kin']:
    if ckpt_compat[name]:
        print(f'[{name}] checkpoint exists and is compatible — skipping training.')
        continue
    path = CKPT_PATHS[name]
    print(f'[{name}] training fresh (wd={GLM_WEIGHT_DECAYS[name]}); will overwrite any existing file ...')
    os.makedirs(os.path.dirname(path), exist_ok=True)
    set_random_state(seed)
    cls, cfg = glm_configs_by_name[name]
    fresh = cls(**cfg).to(device)
    dls, _ = DATA_BY_MODEL[name]
    best_acc, weights, _ = bat_former_trainer(
        model=fresh, dataloaders=dls, device=device,
        n_epochs=1000, lr=DEFAULT_LR,
        weight_decay=GLM_WEIGHT_DECAYS[name],
        loss_fn=nn.PoissonNLLLoss(log_input=False),
        valid_fn=poisson_fde_population, valid_stat='median',
        early_stopping_patience=20,
        save_model=True, model_name=path,
    )
    print(f'  best val FDE (median): {best_acc:.4f}')
    models[name] = fresh

[Transformer (no attn)] checkpoint exists and is compatible — skipping training.
[GLM] training fresh (wd=1.0); will overwrite any existing file ...


Training:  12%|█▏        | 121/1000 [00:24<02:54,  5.03it/s]


Early stopping at epoch 122 (no improvement for 20 epochs)
  best val FDE (median): -0.1008
[GLM + lags] training fresh (wd=10.0); will overwrite any existing file ...
GLM: 11 lags x 4096 spatial = 45056 features, lag_bins=[-10, -8, -6, -4, -2, 0, 2, 4, 6, 8, 10]


Training:   0%|          | 0/1000 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 76.00 MiB. GPU 0 has a total capacity of 31.33 GiB of which 27.44 MiB is free. Process 409940 has 3.93 GiB memory in use. Process 409037 has 20.03 GiB memory in use. Process 1894438 has 734.00 MiB memory in use. Process 807625 has 3.00 GiB memory in use. Including non-PyTorch memory, this process has 1.68 GiB memory in use. Of the allocated memory 941.38 MiB is allocated by PyTorch, and 162.62 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Load all five checkpoints

In [ ]:
for name in MODEL_ORDER:
    path = CKPT_PATHS[name]
    state = torch.load(path, map_location=device)
    missing, unexpected = models[name].load_state_dict(state, strict=True)
    assert not missing and not unexpected, f'{name}: missing={missing}, unexpected={unexpected}'
    models[name] = models[name].to(device).eval()
    print(f'[{name}] loaded {path}')

## Evaluation

Copy of `evaluate_model` from `preprocess-data-v0j-share.ipynb` so this notebook
is self-contained. Each model is evaluated against its routed dataloader
(`DATA_BY_MODEL[name]`): base for everything except `GLM + lags + kin`, which
uses the kinematics dataloader.

In [ ]:
def evaluate_model(model, dataloaders, valid_fn, device, flight_filter=None, meta_info_dict=None):
    """Evaluate model on test data; returns {session_name: per-neuron scores}."""
    results = {}
    model.eval()
    for session_name, session_data in dataloaders['test'].items():
        if flight_filter == 'ood':
            target_types = np.array(meta_info_dict['flight_type_test_ood'][session_name], dtype=int)
            if target_types.size == 0:
                results[session_name] = np.array([])
                continue
        elif flight_filter == 'indomain':
            target_types = np.array(meta_info_dict['flight_type_test_indomain'][session_name], dtype=int)
        else:
            target_types = None

        y_val_list, y_pred_list = [], []
        for batch in session_data:
            X_val = batch.behavior.to(device)
            y_val = batch.spike.to(device)
            mask_val = batch.mask.to(device)
            if target_types is not None:
                batch_mask = np.isin(batch.flight_type_index.numpy(), target_types)
                if not batch_mask.any():
                    continue
                X_val = X_val[batch_mask]
                y_val = y_val[batch_mask]
                mask_val = mask_val[batch_mask]
            y_pred = model(session_name, X_val, mask_val)
            y_val_list.append(y_val[mask_val == 1.0].detach().cpu().numpy())
            y_pred_list.append(y_pred[mask_val == 1.0].detach().cpu().numpy())
        if y_val_list:
            results[session_name] = valid_fn(np.vstack(y_val_list), np.vstack(y_pred_list))
        else:
            results[session_name] = np.array([])
    return results

In [ ]:
metrics = {
    'corr': lambda y_true, y_pred: corr(y_true, y_pred, axis=0),
    'fde':  lambda y_true, y_pred: poisson_fde_population(y_true, y_pred),
}
flight_filters = [None, 'ood', 'indomain']

results_by_model = {name: {} for name in MODEL_ORDER}
for name in MODEL_ORDER:
    dls, mi = DATA_BY_MODEL[name]
    for metric_name, valid_fn in metrics.items():
        for flight_filter in flight_filters:
            key = f'{metric_name}_{flight_filter or "all"}'
            results_by_model[name][key] = evaluate_model(
                models[name], dls, valid_fn, device,
                flight_filter=flight_filter, meta_info_dict=mi,
            )
            all_scores = np.hstack([v for v in results_by_model[name][key].values() if len(v) > 0])
            stat = np.nanmedian if 'fde' in key else np.nanmean
            print(f'  [{name:<22}] {key:<14}: {stat(all_scores):.4f}')

## Long-form DataFrame

One row per (model, session, neuron, metric, flight_filter). The neuron index
within a session is consistent across models (all five share `unit_num_dict` for
non-kin variants; kin uses `unit_num_dict_kin`, which has the same session keys
and the same neuron counts as the base loader).

In [ ]:
rows = []
for model_name, by_key in results_by_model.items():
    for key, by_session in by_key.items():
        metric, flight_filter = key.split('_', 1)
        for session, scores in by_session.items():
            if scores.size == 0:
                continue
            bat = parse_bat(session)
            for unit, score in enumerate(scores):
                rows.append({
                    'model': model_name, 'session': session, 'bat': bat,
                    'unit': unit, 'metric': metric,
                    'flight_filter': flight_filter, 'score': score,
                })
df = pd.DataFrame(rows)
df['model'] = pd.Categorical(df['model'], categories=MODEL_ORDER, ordered=True)
df['flight_filter'] = pd.Categorical(df['flight_filter'], categories=['all', 'indomain', 'ood'], ordered=True)
print(df.shape)
df.head()

In [ ]:
# Wide form (one row per neuron) so we can compute per-neuron deltas + scatters.
wide = df.pivot_table(
    index=['session', 'bat', 'unit', 'metric', 'flight_filter'],
    columns='model', values='score', observed=False,
).reset_index()
wide.head()

## Per-session score, all five models

One paired point per session (median over neurons). Lines colored by bat
(memory rule: always use `bat_palette()`). FDE y-axis clipped to `[-1, 1]` in
the plot only — underlying values stay raw.

In [ ]:
session_df = (
    df.groupby(['metric', 'flight_filter', 'session', 'bat', 'model'], observed=True)
      .agg(score=('score', 'median'))
      .reset_index()
)

for metric in ['corr', 'fde']:
    g = sns.FacetGrid(
        session_df[session_df['metric'] == metric],
        col='flight_filter', col_order=['all', 'indomain', 'ood'],
        height=3.6, aspect=1.15, sharex=True, sharey=True,
    )
    g.map_dataframe(
        sns.lineplot, x='model', y='score', units='session',
        hue='bat', palette=BAT_PALETTE, hue_order=list(BAT_PALETTE.keys()),
        estimator=None, marker='o', alpha=0.7, linewidth=1, legend=False,
    )
    for flight_filter, ax in g.axes_dict.items():
        # Black line connecting per-model session-medians shows the trend.
        sub = (session_df[(session_df['metric'] == metric) &
                          (session_df['flight_filter'] == flight_filter)]
               .groupby('model', observed=True)['score'].median()
               .reindex(MODEL_ORDER))
        ax.plot(range(len(sub)), sub.values, color='k', linewidth=2.5, marker='s',
                markersize=7, label='cross-session median', zorder=5)
        ax.tick_params(axis='x', rotation=30)
        for lbl in ax.get_xticklabels():
            lbl.set_horizontalalignment('right')
        if metric == 'fde':
            ax.set_ylim(-1, 1)
    g.set_axis_labels('', f'session-median {metric.upper()}')
    g.figure.suptitle(f'{metric.upper()} per session, across all five models', y=1.04)
    plt.show()

## Adjacent-model pairwise significance

Wilcoxon signed-rank on the per-session-median scores between adjacent models in
`MODEL_ORDER` (memory rule: session-level nonparametric stats by default).
Bracket + stars shown only when significant (3 tiers, hide n.s.).

In [ ]:
session_wide = session_df.pivot_table(
    index=['metric', 'flight_filter', 'session', 'bat'],
    columns='model', values='score', observed=False,
).reset_index()

adjacent_pairs = list(zip(MODEL_ORDER[:-1], MODEL_ORDER[1:]))

# Boxplot per metric x flight_filter, with adjacent-pair brackets.
for metric in ['corr', 'fde']:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
    for ax, flight_filter in zip(axes, ['all', 'indomain', 'ood']):
        sub_long = session_df[(session_df['metric'] == metric) &
                              (session_df['flight_filter'] == flight_filter)]
        sns.boxplot(data=sub_long, x='model', y='score', order=MODEL_ORDER,
                    color='lightgray', showfliers=False, ax=ax)
        sns.stripplot(data=sub_long, x='model', y='score', order=MODEL_ORDER,
                      hue='bat', palette=BAT_PALETTE, hue_order=list(BAT_PALETTE.keys()),
                      dodge=False, jitter=0.15, size=5, alpha=0.85,
                      edgecolor='k', linewidth=0.3, ax=ax, legend=False)
        sub_wide = session_wide[(session_wide['metric'] == metric) &
                                (session_wide['flight_filter'] == flight_filter)]
        for i, (m_a, m_b) in enumerate(adjacent_pairs):
            paired = sub_wide[[m_a, m_b]].dropna()
            if len(paired) < 3:
                continue
            _, p = wilcoxon_safe((paired[m_b] - paired[m_a]).values)
            add_significance_bracket(ax, i, i + 1, p)
        if metric == 'fde':
            ax.set_ylim(-1, 1)
        ax.tick_params(axis='x', rotation=30)
        for lbl in ax.get_xticklabels():
            lbl.set_horizontalalignment('right')
        ax.set_xlabel('')
        ax.set_ylabel(f'session-median {metric.upper()}')
        ax.set_title(f'flight_filter = {flight_filter}')
    fig.suptitle(f'{metric.upper()}: adjacent-pair Wilcoxon (session-level)', y=1.02)
    plt.tight_layout()
    plt.show()

## Summary tables

Per-model median across neurons (or sessions) and the full pairwise Wilcoxon
matrix on per-session medians.

In [ ]:
neuron_summary = (
    df.groupby(['metric', 'flight_filter', 'model'], observed=True)
      .agg(n_neurons=('score', 'size'),
           neuron_median=('score', 'median'),
           neuron_mean=('score', 'mean'))
      .reset_index()
)
neuron_summary

In [ ]:
# Pairwise Wilcoxon signed-rank on per-session medians.
# Alternative: H1 = column model > row model. p < 0.05 means column beats row.
def stars(p):
    if not np.isfinite(p): return ''
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'

pair_rows = []
for metric in ['corr', 'fde']:
    for flight_filter in ['all', 'indomain', 'ood']:
        sub = session_wide[(session_wide['metric'] == metric) &
                           (session_wide['flight_filter'] == flight_filter)]
        for m_a in MODEL_ORDER:
            for m_b in MODEL_ORDER:
                if m_a == m_b:
                    continue
                paired = sub[[m_a, m_b]].dropna()
                if len(paired) < 3:
                    continue
                diff = (paired[m_b] - paired[m_a]).values
                try:
                    _, p = wilcoxon(diff, alternative='greater')
                except ValueError:
                    p = np.nan
                pair_rows.append({
                    'metric': metric, 'flight_filter': flight_filter,
                    'row (baseline)': m_a, 'col (alternative)': m_b,
                    'n_sessions': len(paired),
                    'median_diff (col - row)': float(np.median(diff)),
                    'p (col > row)': p, 'sig': stars(p),
                })
pair_df = pd.DataFrame(pair_rows)
print('Adjacent-pair summary:')
adj = pair_df[pair_df.apply(
    lambda r: (r['row (baseline)'], r['col (alternative)']) in adjacent_pairs, axis=1
)]
adj[['metric', 'flight_filter', 'row (baseline)', 'col (alternative)',
     'n_sessions', 'median_diff (col - row)', 'p (col > row)', 'sig']]